# Week 51, Day 3: Models of the block prior: a Bayesian observer against simple kernels

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w051/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.special import digamma
d = np.load(dataset("ibl_biased.npz"))
mouse, session = d["mouse"].astype(int), d["session"].astype(int)
contrast, choice, reward = d["contrast"].astype(float), d["choice"].astype(float), d["reward"].astype(int)
p_left, rt = d["p_left"].astype(float).round(2), d["rt"].astype(float)
ok = (choice >= 0) & d["included"] & np.isfinite(rt)
sess = 10 * mouse + session
f_c = np.tanh(5 * contrast) / np.tanh(5)
c_pm = np.where(choice >= 0, 2 * choice - 1, 0.0)
# the stimulus side, +1 right and -1 left; at zero contrast it is revealed by the feedback
side = np.where(contrast != 0, np.sign(contrast), np.where(reward == 1, c_pm, -c_pm))

def kernel(x, alpha, sess_ids):
    """pi_t = (1 - alpha) pi_{t-1} + alpha x_{t-1}, restarted at zero in every session."""
    out = np.zeros(len(x))
    for s in np.unique(sess_ids):
        k = sess_ids == s
        out[k] = lfilter([alpha], [1, -(1 - alpha)], np.r_[0, x[k][:-1]])
    return out
print(f"stimulus side known on {np.mean(side != 0):.2%} of trials")

### Exercise: The Bayesian block observer

Complete bayes_prior. Before each trial, record the log-odds of a right stimulus; then update the belief b = [P(right block), P(left block)] with the likelihood of the observed side (0.8 and 0.2, or 0.2 and 0.8), normalise, and apply the switch probability h.

In [ ]:
def bayes_prior(side, sess_ids, h=0.02):
    out = np.zeros(len(side))
    for s in np.unique(sess_ids):
        b = np.array([0.5, 0.5])                       # belief at the start of a session
        for t in np.flatnonzero(sess_ids == s):
            pr = 0.8 * b[0] + 0.2 * b[1]               # predicted P(stimulus on the right)
            out[t] = ...
            like = np.array([0.8, 0.2]) if side[t] > 0 else np.array([0.2, 0.8])
            b = ...                                     # Bayes' rule, normalised
            b = ...                                     # a switch may occur before the next trial
    return out

prior_bayes = bayes_prior(side, sess)
for p in [0.2, 0.5, 0.8]:
    print(f"p_left = {p}: mean Bayesian prior {prior_bayes[p_left == p].mean():+.2f}")

In [ ]:
import numpy as np
def test_first():
    "the prior is zero on the first trial of every session"
    first = np.r_[True, sess[1:] != sess[:-1]]
    assert np.allclose(prior_bayes[first], 0)
def test_range():
    "the prior log-odds stays within plus and minus log 4"
    assert np.all(np.abs(prior_bayes) <= np.log(4) + 1e-9)
def test_update():
    "after three right stimuli from an even belief, the prior is as computed by hand"
    b = np.array([0.5, 0.5])
    for _ in range(3):
        b = b * np.array([0.8, 0.2]); b = b / b.sum(); b = np.array([0.98 * b[0] + 0.02 * b[1], 0.98 * b[1] + 0.02 * b[0]])
    pr = 0.8 * b[0] + 0.2 * b[1]
    out = bayes_prior(np.array([1.0, 1, 1, 1]), np.zeros(4, int))
    assert abs(out[3] - np.log(pr / (1 - pr))) < 1e-9
def test_tracks():
    "the prior is higher in right blocks than in left blocks"
    assert prior_bayes[p_left == 0.2].mean() > 0.5 and prior_bayes[p_left == 0.8].mean() < -0.5

test_first()
test_range()
test_update()
test_tracks()
print("All checks passed")

### Exercise: Four models, 18 mice

Complete nll so that it builds the logit for each model (add w_p times the model's prior to w_c f_c + b) and returns the negative Bernoulli log-likelihood over the selected trials. Then store the maximised log-likelihood in lme_raw and -BIC/2 in lme.

In [ ]:
models = ["none", "bayes", "stimulus", "action"]
x0 = {"none": [3, 0], "bayes": [3, 0, 1], "stimulus": [3, 0, 1, 0.2], "action": [3, 0, 1, 0.2]}
bounds = {"none": [(-20, 20)] * 2, "bayes": [(-20, 20)] * 3,
          "stimulus": [(-20, 20)] * 3 + [(0.01, 1)], "action": [(-20, 20)] * 3 + [(0.01, 1)]}

def logit(model, p, m):
    """The logit of P(right) on every trial of mouse m (all trials, usable or not)."""
    k = mouse == m
    z = p[0] * f_c[k] + p[1]
    if model == "bayes":
        z = z + p[2] * prior_bayes[k]
    if model == "stimulus":
        z = z + ...
    if model == "action":
        z = z + ...
    return z

def nll(p, model, m):
    use = ok[mouse == m]
    z, y = logit(model, p, m)[use], choice[mouse == m][use]
    return ...

lme_raw, lme, params = np.zeros((18, 4)), np.zeros((18, 4)), {}
for m in range(18):
    n = np.sum(ok & (mouse == m))
    for j, mod in enumerate(models):
        res = minimize(nll, x0[mod], args=(mod, m), method="L-BFGS-B", bounds=bounds[mod])
        params[m, mod] = res.x
        lme_raw[m, j] = ...
        lme[m, j] = ...
best = lme.argmax(1)
print("models with the most evidence:", {mod: int(np.sum(best == j)) for j, mod in enumerate(models)})
print("summed log evidence relative to the best:", np.round(lme.sum(0) - lme.sum(0).max(), 1))

In [ ]:
import numpy as np
def test_shape():
    "lme has one row per mouse and one column per model"
    assert lme.shape == (18, 4) and np.all(np.isfinite(lme))
def test_nested():
    "every model with a prior fits at least as well as no prior"
    assert np.all(lme_raw[:, 1:] >= lme_raw[:, :1] - 1e-6)
def test_winner():
    "the action kernel has the most evidence for at least 15 mice"
    assert np.sum(lme.argmax(1) == 3) >= 15

test_shape()
test_nested()
test_winner()
print("All checks passed")

### Exercise: How common is each kind of learner?

Run rfx_bms on lme, convert alpha into expected frequencies, and estimate the exceedance probabilities: draw 20,000 samples from Dirichlet(alpha) with a seeded generator and count how often each model has the largest frequency.

In [ ]:
def rfx_bms(lme, iters=200):
    a0 = np.ones(lme.shape[1]); a = a0.copy()
    for _ in range(iters):
        lu = lme + digamma(a) - digamma(a.sum()); lu -= lu.max(1, keepdims=True)
        u = np.exp(lu); g = u / u.sum(1, keepdims=True); a = a0 + g.sum(0)
    return a, g

alpha_d, g = rfx_bms(lme)
freq = ...
samples = np.random.default_rng(0).dirichlet(alpha_d, 20000)
xp = ...
for mod, f, x in zip(models, freq, xp):
    print(f"{mod:9s} expected frequency {f:.3f}, exceedance probability {x:.3f}")

In [ ]:
import numpy as np
def test_freq():
    "freq is the expected frequency of each model and sums to 1"
    assert freq.shape == (4,) and abs(freq.sum() - 1) < 1e-9 and freq[3] > 0.7
def test_xp():
    "xp is estimated from 20,000 Dirichlet samples and the action kernel's exceeds 0.99"
    assert abs(xp.sum() - 1) < 1e-9 and xp[3] > 0.99

test_freq()
test_xp()
print("All checks passed")

### Exercise: Rates and weights

Collect the action model's rate and prior weight for each mouse into alphas and wps. Compute zero_shift, each mouse's P(right) at zero contrast in right blocks minus that in left blocks, and its correlation r with wps.

In [ ]:
alphas = ...
wps = ...
zero_shift = np.zeros(18)
for m in range(18):
    k = ok & (contrast == 0) & (mouse == m)
    zero_shift[m] = ...
r = ...
print(f"rate alpha: median {np.median(alphas):.2f} (range {alphas.min():.2f} to {alphas.max():.2f}); timescale about {1 / np.median(alphas):.1f} trials")
print(f"prior weight w_p: median {np.median(wps):.2f}")
print(f"correlation between w_p and the zero-contrast block effect: r = {r:.2f}")
plt.scatter(wps, zero_shift); plt.xlabel("fitted prior weight w_p"); plt.ylabel("zero-contrast block effect"); plt.show()

In [ ]:
import numpy as np
def test_arrays():
    "alphas and wps hold the action model's rate and prior weight for each mouse"
    assert np.allclose(alphas, [params[m, "action"][3] for m in range(18)]) and np.allclose(wps, [params[m, "action"][2] for m in range(18)])
def test_shift():
    "zero_shift is each mouse's P(right) at zero contrast in right blocks minus left blocks"
    k = ok & (contrast == 0) & (mouse == 4)
    exp = choice[k & (p_left == 0.2)].mean() - choice[k & (p_left == 0.8)].mean()
    assert abs(zero_shift[4] - exp) < 1e-12
def test_r():
    "r is the correlation across mice between wps and zero_shift"
    assert abs(r - np.corrcoef(wps, zero_shift)[0, 1]) < 1e-12 and r > 0.4

test_arrays()
test_shift()
test_r()
print("All checks passed")